In [7]:
import os
import time
import pickle
import warnings
import csv
from datetime import datetime, timedelta, date

import requests
import numpy as np
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from scipy.stats import randint, uniform

from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

from category_encoders import TargetEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# Ophalen data

python version: 3.11.9

## Weer api

In [8]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.05,
    "longitude": 3.7167,
    "start_date": "2016-10-01",
    "end_date": "2025-04-10",
    "hourly": "temperature_2m,apparent_temperature,rain,snowfall,weather_code,cloud_cover,wind_speed_10m,sunshine_duration",
    "timezone": "auto"
}

In [9]:
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    hourly_data = data.get("hourly", {})
    script_dir = os.getcwd()
    csv_file_path = os.path.join(script_dir, ".", "data csv", "weerdataraw.csv")
    os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)

    #open csv
    with open(csv_file_path, mode="w", newline="") as file:
        writer = csv.writer(file)

        #header csv
        header = ["timestamp", "temperature_2m", "apparent_temperature", "rain", "snowfall", "weather_code", "cloud_cover", "wind_speed_10m", "sunshine_duration"]
        writer.writerow(header)

        #naar csv schrijven
        for i in range(len(hourly_data.get("temperature_2m", []))):
            row = [
                hourly_data.get("time", [])[i],
                hourly_data.get("temperature_2m", [])[i],
                hourly_data.get("apparent_temperature", [])[i],
                hourly_data.get("rain", [])[i],
                hourly_data.get("snowfall", [])[i],
                hourly_data.get("weather_code", [])[i],
                hourly_data.get("cloud_cover", [])[i],
                hourly_data.get("wind_speed_10m", [])[i],
                hourly_data.get("sunshine_duration", [])[i]
            ]
            writer.writerow(row)

    print(f"Data weggeschreven: {csv_file_path}")
else:
    print(f"Failed status code: {response.status_code}")

Data weggeschreven: c:\Users\krist\OneDrive\Documents\HoGent IT\ML_kijkcijfers\ML_project_kijkcijfers\.\data csv\weerdataraw.csv


## Kijkcijfer data

In [10]:
start_date = datetime(2016, 10, 1)
end_date = datetime.today()

script_dir = os.getcwd()
output_file = os.path.join(script_dir, "./data csv/kijkcijfersdataraw.csv")

os.makedirs(os.path.dirname(output_file), exist_ok=True)

In [11]:

with open(output_file, mode='w', newline='', encoding='utf-8-sig') as file:
    fieldnames = [
    'dateDiff', 'ranking', 'description', 'channel', 
    'startTime', 'rLength', 'rateInK', 'live'
    ]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    # Loop door elke dag
    current_date = start_date
    while current_date <= end_date:
        datum = f"{current_date.year}-{current_date.month}-{current_date.day}"
        url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"
            
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                programma_lijst = data.get('hydra:member', [])
                        
                for programma in programma_lijst:
                    try:
                        writer.writerow({
                        'dateDiff': programma.get('dateDiff'),
                        'ranking': programma.get('ranking'),
                        'description': programma.get('description'),
                        'channel': programma.get('channel'),
                        'startTime': programma.get('startTime'),
                        'rLength': programma.get('rLength'),
                        'rateInK': programma.get('rateInK'),
                        'live': programma.get('live')
                        })
                                    
                    except Exception as e:
                        print(f"error {datum}: {e}")         
            else:
                print(f"no data {datum}")
                        
        except Exception as e:
            print(f"error {datum}: {e}")
            
        current_date += timedelta(days=1)
print("data opgehaald")

data opgehaald


# Opkuisen data

In [18]:
def cleanUp(df):
  #nullwaarden verwijderen
  df.dropna(inplace=True)

  # foute namen van kolom Kanaal corrigeren
  df = df[~df['channel'].str.contains(r'[,/]', regex=True, na=False)]

  #tijd aanpassen
  #tijdformaat ..:..:..
  tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
  #omzetten naar datetime
  df['date'] = pd.to_datetime(df['dateDiff']).dt.date
  #filter rijen met formaat
  df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
  #afleveringlengte naar seconden omzetten 
  df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)
  #kijkers naar int
  df['Kijkers'] = df['rateInK'].dropna().astype(str).str.replace('.', '', regex=False).astype(int)

  #uren met 24+
  def time_cor(rij):
    tijdArr = rij['startTime'].split(':')
    if int(tijdArr[0]) >= 24:
      tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
      
      rij['date'] += timedelta(days=1)
    rij['startTime'] = ':'.join(tijdArr)
    return rij
  
  df = df.apply(time_cor, axis=1)

  #1 kolom voor beide data
  df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                  + " " + df['startTime'].astype(str))
  
  #hour en minute voor join later on
  df['hour'] = pd.to_datetime(df['startTime']).dt.hour
  df['minute'] = pd.to_datetime(df['startTime']).dt.minute

  #kolommen verwijderen die niet nodig meer zijn
  df.drop(['startTime', 'rLength', 'rateInK', 'ranking', 'live'], axis=1, inplace=True)

  #de nieuwe dataframe
  df = df[['FullDate', 'date', 'hour','minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

  #hernoemen kolommen
  df.rename(columns={'description':'Programma', 'channel':'Kanaal'}, inplace=True)

  return df

In [19]:
Kijkcijfers = pd.read_csv('./data csv/kijkcijfersdataraw.csv')
Kijkcijfers = cleanUp(Kijkcijfers)
Kijkcijfers.sample(10)

,FullDate,date,hour,minute,Kanaal,Programma,Lengte_sec,Kijkers
19655,2019-06-11 18:29:55,2019-06-11,18,29,EEN,BLOKKEN,1617,494742
33024,2021-04-11 21:04:27,2021-04-11,21,4,VTM,"SERGIO & AXEL, VAN DE KAART",3443,217197
38051,2021-12-19 19:00:02,2021-12-19,19,0,EEN,HET 7 UUR-JOURNAAL,2211,962358
11052,2018-04-08 20:32:41,2018-04-08,20,32,EEN,DE 3 WIJZEN,2828,790760
47001,2023-02-21 20:43:42,2023-02-21,20,43,EEN,HET HOGE NOORDEN,3083,885874
26233,2020-05-07 22:09:05,2020-05-07,22,9,EEN,VANDAAG,3160,494395
18095,2019-03-25 22:11:17,2019-03-25,22,11,EEN,VAN GILS & GASTEN,2884,559735
51665,2023-10-12 18:29:34,2023-10-12,18,29,VRT 1,BLOKKEN,1633,580250
243,2016-10-13 19:44:07,2016-10-13,19,44,EEN,IEDEREEN BEROEMD,1286,899255
24728,2020-02-22 20:14:42,2020-02-22,20,14,EEN,FC DE KAMPIOENEN,1805,675707


# Mergen van kijkcijfers en weerdata

In [20]:
weerData = pd.read_csv('./data csv/weerdataraw.csv')
weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
#naar zelfde formaat als kijkcijfer datum
weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

#hour voor join later on
weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

#verwijder kolom
weerData = weerData.drop(columns=['timestamp'])

weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                           'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                           'wind_speed_10m', 'sunshine_duration']]

#hernoemen kolommen
weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'wind_speed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

weerData.sample(10)

,datetime,date,hour,minute,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn
29809,2020-02-25 01:00:00,2020-02-25,1,0,10.2,5.6,0.0,0.0,3,100,28.1,0.0
39988,2021-04-24 04:00:00,2021-04-24,4,0,4.2,0.9,0.0,0.0,3,94,11.1,0.0
12837,2018-03-19 21:00:00,2018-03-19,21,0,0.3,-5.1,0.0,0.0,0,0,16.4,0.0
1671,2016-12-09 15:00:00,2016-12-09,15,0,11.1,8.3,0.0,0.0,0,5,15.7,3600.0
59066,2023-06-28 02:00:00,2023-06-28,2,0,16.2,15.0,0.0,0.0,3,100,11.2,0.0
46521,2022-01-21 09:00:00,2022-01-21,9,0,3.1,0.2,0.0,0.0,1,41,7.4,0.0
71601,2024-12-01 09:00:00,2024-12-01,9,0,2.1,-1.9,0.0,0.0,3,98,12.8,0.0
12545,2018-03-07 17:00:00,2018-03-07,17,0,7.7,4.3,1.2,0.0,55,77,15.8,3600.0
56320,2023-03-05 16:00:00,2023-03-05,16,0,4.7,0.4,0.3,0.0,51,100,16.8,0.0
51251,2022-08-06 11:00:00,2022-08-06,11,0,19.1,17.4,0.0,0.0,3,100,10.0,3600.0


In [21]:
kijkcijfersWeer = pd.merge(Kijkcijfers, weerData, on=['date', 'hour'], how='left')

kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]

kijkcijfersWeer.sample(10)

#kijkcijfersWeer.to_csv('./data csv/blablableble.csv')

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn
51846,2023-12-04 20:15:49,2023-12-04,20,VRT 1,THUIS,1491,1109023,5.0,1.6,0.7,0.0,53.0,100.0,13.1,0.00
1593,2016-12-22 20:31:53,2016-12-22,20,CANVAS,DE AFSPRAAK,2350,243378,6.5,4.2,0.0,0.0,3.0,100.0,9.2,0.00
47509,2023-04-30 19:24:17,2023-04-30,19,EEN,SPORTWEEKEND,1851,543443,16.4,14.9,0.0,0.0,3.0,100.0,6.5,2691.77
53908,2024-03-17 19:23:12,2024-03-17,19,VRT 1,SPORTWEEKEND,1987,850301,12.4,9.5,0.2,0.0,51.0,86.0,18.8,0.00
23868,2020-01-14 21:45:27,2020-01-14,21,VTM,TELEFACTS WINTER,2590,416636,11.6,5.1,0.8,0.0,53.0,94.0,39.6,0.00
51407,2023-11-12 20:00:25,2023-11-12,20,VTM,DE VERRADERS,4711,915392,6.3,3.9,0.0,0.0,3.0,98.0,8.5,0.00
29516,2020-11-06 20:00:04,2020-11-06,20,Canvas,TER ZAKE,1852,243080,6.5,2.8,0.0,0.0,0.0,0.0,14.4,0.00
9233,2018-01-10 22:22:56,2018-01-10,22,VTM,TELEFACTS,2426,273491,6.2,3.5,0.0,0.0,3.0,85.0,10.2,0.00
18834,2019-05-05 18:12:54,2019-05-05,18,EEN,DAGELIJKSE KOST,2725,244849,9.9,4.6,0.0,0.0,2.0,63.0,22.5,3600.00
2477,2017-02-04 20:10:53,2017-02-04,20,CANVAS,VRANCKX,1401,195458,4.9,2.4,0.0,0.0,3.0,100.0,7.4,0.00


In [22]:
kijkcijfersWeer.dropna(inplace=True)
#csv aanmaken als het niet bestaat
csv_file_path = os.path.join('.','data csv','kijkcijfersWeerRaw.csv')
os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)
#data naar csv
kijkcijfersWeer.to_csv(csv_file_path, index=False)

# Features 

Feestdag, day of the week, weekend, seizoen

In [23]:
cleanData = pd.read_csv('./data csv/kijkcijfersWeerRaw.csv')
cleanData['date'] = pd.to_datetime(cleanData['date'])

#feestdagen
feestdagen = holidays.BE()
cleanData['isFeestdag'] = cleanData['date'].apply(lambda x: 1 if x in feestdagen else 0)

#dag van de week
cleanData['Weekdag'] = cleanData['date'].dt.weekday

#weekend
cleanData['isWeekend'] = cleanData['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

cleanData['Seizoen'] = cleanData['date'].apply(seizoenFinder)
cleanData.sample(10)

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
9311,2018-01-14 20:39:24,2018-01-14,20,VTM,D'ARDENNEN,5138,418669,2.3,-0.7,0.0,0.0,1.0,38.0,7.2,0.00,0,6,1,winter
43922,2022-10-30 12:59:39,2022-10-30,12,VTM,NIEUWS 13U VTM,2147,195514,17.4,18.1,0.0,0.0,3.0,100.0,5.4,3600.00,0,6,1,herfst
1696,2016-12-27 12:59:49,2016-12-27,12,VTM,NIEUWS 13U VTM,2015,187029,5.1,2.4,0.0,0.0,0.0,2.0,8.6,3600.00,0,1,0,winter
32224,2021-03-23 20:03:22,2021-03-23,20,VTM,FAMILIE,1538,607504,7.8,5.5,0.0,0.0,1.0,36.0,7.7,145.71,0,1,0,lente
25758,2020-04-18 19:00:04,2020-04-18,19,EEN,HET 7 UUR-JOURNAAL,2844,1261510,12.9,11.0,0.0,0.0,1.0,49.0,10.4,3600.00,0,5,1,lente
29699,2020-11-16 20:19:11,2020-11-16,20,EEN,THUIS,1354,1215112,10.6,7.6,0.0,0.0,3.0,100.0,14.1,0.00,0,0,0,herfst
18268,2019-04-07 21:57:38,2019-04-07,21,EEN,JOANNA LUMLEY'S INDIA,2758,324479,12.0,10.8,0.0,0.0,2.0,69.0,10.9,0.00,0,6,1,lente
23851,2020-01-13 13:00:05,2020-01-13,13,EEN,HET 1 UUR-JOURNAAL,1791,344080,7.6,3.6,0.0,0.0,1.0,40.0,19.8,3600.00,0,0,0,winter
59803,2025-01-20 20:00:03,2025-01-20,20,VRT CANVAS,TER ZAKE,2150,204899,0.4,-2.9,0.1,0.0,51.0,100.0,7.5,0.00,0,0,0,winter
56242,2024-07-22 20:33:16,2024-07-22,20,VTM2,"ALERT, MISSING PERSONS UNIT",2471,159456,21.3,20.9,0.0,0.0,3.0,100.0,15.6,62.01,0,0,0,zomer


In [24]:
script_dir = os.getcwd()
csv_file_path = os.path.join(script_dir, ".", "data csv", "kijkcijfersWeer.csv")
os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)
cleanData.to_csv('./data csv/kijkcijfersWeer.csv')

# Model training

In [25]:
cleanData = pd.read_csv('./data csv/kijkcijfersWeer.csv')
cleanData.sample(5)

,Unnamed: 0,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
6434,6434,2017-08-22 12:59:48,2017-08-22,12,VTM,NIEUWS 13U VTM,2031,186456,21.3,21.8,0.0,0.0,3.0,91.0,5.7,3600.00,0,1,0,zomer
13319,13319,2018-08-03 20:06:16,2018-08-03,20,EEN,FC DE KAMPIOENEN,2285,607519,27.8,26.5,0.0,0.0,0.0,0.0,11.8,3600.00,0,4,0,zomer
15484,15484,2018-11-19 20:39:22,2018-11-19,20,VTM,HOE ZAL IK HET ZEGGEN?,2892,707088,3.4,-2.2,0.0,0.0,0.0,2.0,24.5,0.00,0,0,0,herfst
4144,4144,2017-04-29 17:08:37,2017-04-29,17,VTM,TOM SAYS THANKS!,5952,321030,13.2,9.7,0.0,0.0,2.0,73.0,12.2,699.79,0,5,1,lente
17965,17965,2019-03-23 20:14:19,2019-03-23,20,EEN,ANIMAL AIRPORT,1405,440592,8.9,5.9,0.0,0.0,3.0,90.0,13.2,0.00,0,5,1,lente


### lag features toevoegen

In [26]:
def lagFeatures(df, n):
  for i in range(1,n+1):
    df[f'KijkersLag{i}'] = df.sort_values('FullDate').groupby('Programma')['Kijkers'].shift(i).fillna(df.groupby('Programma')['Kijkers'].transform('mean'))
  return df

In [27]:
cleanData = lagFeatures(cleanData, 3)
cleanData['Kanaal'] = cleanData['Kanaal'].str.replace(' ', '_')

print(cleanData['Kanaal'].unique())

cleanData[cleanData['Programma'] == 'THUIS'][['FullDate', 'Kijkers', 'KijkersLag1', 'KijkersLag2', 'KijkersLag3']]

['EEN' 'VTM' 'CANVAS' 'VIER' 'VIJF' 'Q2' 'VITAYA' 'OP_12' 'ZES' 'LA_UNE'
 'RTL-TVI' 'TF1' 'AB3' 'KETNET' 'Canvas' 'CAZ' 'VTM2'
 'ELEVEN_PRO_LEAGUE_1_NL' 'VTM3' 'VTM4' 'PLAY_SPORTS_OPEN' 'PLAY4' 'PLAY5'
 'EUROSPORT_1_(NL)' 'PLAY6' 'VRT_1' 'VRT_CANVAS' 'VTM_GOLD'
 'DAZN_PRO_LEAGUE_1_(NL)']


,FullDate,Kijkers,KijkersLag1,KijkersLag2,KijkersLag3
40,2016-10-03 20:14:23,1268561,1.060382e+06,1.060382e+06,1.060382e+06
60,2016-10-04 20:09:27,1169791,1.268561e+06,1.060382e+06,1.060382e+06
80,2016-10-05 20:13:52,1244502,1.169791e+06,1.268561e+06,1.060382e+06
121,2016-10-07 19:57:34,1156477,1.244502e+06,1.169791e+06,1.268561e+06
180,2016-10-10 19:59:00,1315826,1.156477e+06,1.244502e+06,1.169791e+06
...,...,...,...,...,...
61265,2025-04-04 20:24:13,956032,1.029899e+06,9.989180e+05,1.074137e+06
61325,2025-04-07 20:14:35,1021508,9.560320e+05,1.029899e+06,9.989180e+05
61345,2025-04-08 20:13:09,1014480,1.021508e+06,9.560320e+05,1.029899e+06
61365,2025-04-09 20:19:53,953207,1.014480e+06,1.021508e+06,9.560320e+05


## Encoding

In [28]:
#kardinaliteiten voor betere toepassingen
for kolom in cleanData.columns:
  print(f'{kolom}: {cleanData[kolom].nunique()}')

Unnamed: 0: 61405
FullDate: 61318
date: 3051
hour: 21
Kanaal: 29
Programma: 5961
Lengte_sec: 6774
Kijkers: 58864
Temperatuur: 414
Gevoelstemp: 478
Regen: 62
Sneeuw: 25
Weercode: 13
Bewolking: 101
Windsnelheid: 456
Zonnenschijn: 3555
isFeestdag: 2
Weekdag: 7
isWeekend: 2
Seizoen: 4
KijkersLag1: 59000
KijkersLag2: 56717
KijkersLag3: 55153


#### one hot encoding

In [29]:
#one hot encoding voor lage cardinaliteit
oneHotEnc = OneHotEncoder(handle_unknown="ignore")
lageKardinaliteit = cleanData[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
oneHot = oneHotEnc.fit_transform(lageKardinaliteit)
dfOneHot = pd.DataFrame(oneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKardinaliteit.index)

cleanData = cleanData.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
cleanData = pd.concat([cleanData, dfOneHot], axis=1)

cleanData.sample(5)

,Unnamed: 0,FullDate,date,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,...,Weekdag_1,Weekdag_2,Weekdag_3,Weekdag_4,Weekdag_5,Weekdag_6,Seizoen_herfst,Seizoen_lente,Seizoen_winter,Seizoen_zomer
52137,52137,2023-12-18 20:00:02,2023-12-18,DE TAFEL VAN GERT,4165,281885,5.2,0.6,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
59104,59104,2024-12-16 20:00:03,2024-12-16,TER ZAKE,2074,157006,9.0,6.4,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
51546,51546,2023-11-19 20:07:53,2023-11-19,DOWN THE ROAD,3247,1275843,11.8,7.2,0.4,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
45049,45049,2022-12-28 19:00:02,2022-12-28,HET 7 UUR-JOURNAAL,2365,784487,9.6,3.8,0.3,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
31535,31535,2021-02-16 12:59:53,2021-02-16,NIEUWS 13U VTM,1845,236462,8.3,3.5,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [30]:
#opslaan bij models
modelsDir = os.getcwd()
modelFile = os.path.join(modelsDir, ".",
                          "models", "oneHotEncoder.pkl")
os.makedirs(os.path.dirname(modelFile), exist_ok=True)
with open(modelFile, 'wb') as m:
  pickle.dump(oneHotEnc, m)

### target encoding

In [31]:
#target encoding voor medium kardinaliteiten
targetEncoding = TargetEncoder()
medKardinaliteit = cleanData[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
#verdere feature engineering op vorig model
target = targetEncoding.fit_transform(medKardinaliteit, cleanData['Programma'])
target.sample(10)

,date,Programma,Lengte_sec,Temperatuur,Gevoelstemp,Regen,Bewolking,Windsnelheid,Zonnenschijn
15914,2015.89131,3418.000000,2002,6.1,2.4,0.0,64.0,12.3,0.00
47042,2132.74131,1850.000002,2689,9.6,6.6,0.7,100.0,18.0,0.00
773,1991.19131,1749.000000,978,6.3,3.3,0.0,81.0,11.6,0.00
37196,2080.31631,2031.250326,13574,4.1,0.8,1.0,100.0,12.0,0.00
49228,2299.11631,3365.000000,1852,19.8,17.8,0.0,100.0,14.0,2693.49
40662,2096.61522,2215.024167,2800,24.0,21.9,0.0,100.0,10.6,695.81
45282,2042.66631,1726.492923,3864,9.5,5.2,0.0,83.0,22.8,3600.00
23493,2357.91631,2548.000000,1573,4.7,0.8,0.0,100.0,15.0,0.00
28683,2304.99131,1878.000000,1793,13.1,11.4,0.0,100.0,11.8,3381.46
55294,2002.99131,864.623858,2879,15.8,15.2,0.0,13.0,7.4,3600.00


In [32]:
cleanData = cleanData.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
cleanData = pd.concat([cleanData, target], axis=1)
cleanData.sample(5)

,Unnamed: 0,FullDate,Kijkers,Sneeuw,Weercode,isWeekend,KijkersLag1,KijkersLag2,KijkersLag3,hour_0,...,Seizoen_zomer,date,Programma,Lengte_sec,Temperatuur,Gevoelstemp,Regen,Bewolking,Windsnelheid,Zonnenschijn
21156,21156,2019-08-31 18:19:32,133064,0.0,3.0,1,185968.0,191481.0,252278.0,0.0,...,1.0,2475.466310,1294.000460,1542,26.6,26.7,0.0,80.0,12.7,3600.00
39807,39807,2022-04-02 20:40:24,214739,0.0,2.0,1,153176.0,205687.0,213783.0,0.0,...,0.0,2200.561197,2125.547628,2705,5.0,-0.3,0.0,51.0,16.6,0.00
10733,10733,2018-03-26 20:36:07,271523,0.0,0.0,0,224831.0,254090.0,300746.0,0.0,...,0.0,1999.816310,883.000000,2492,4.3,1.4,0.0,2.0,8.7,2163.89
21281,21281,2019-09-07 19:53:44,490820,0.0,1.0,1,335705.0,288256.0,387906.0,0.0,...,1.0,2355.916310,878.000496,1554,15.3,12.2,0.0,41.0,21.4,3600.00
40033,40033,2022-04-11 21:18:39,189400,0.0,3.0,0,107549.0,152331.0,268345.0,0.0,...,0.0,2072.832543,3335.870969,2969,12.5,9.2,0.0,97.0,13.9,0.00


### Verbanden

In [33]:
dataOneHotTarget = cleanData.select_dtypes(include=[np.number])

verband = dataOneHotTarget.corr()

verband = verband['Kijkers'].abs().sort_values(ascending=False)

print(f'{verband.head(10)}')
dataOneHotTarget.shape

Kijkers        1.000000
KijkersLag1    0.894654
KijkersLag2    0.893278
KijkersLag3    0.876085
hour_19        0.386031
Kanaal_EEN     0.332882
hour_22        0.184443
hour_20        0.176510
hour_17        0.176477
Gevoelstemp    0.172559
Name: Kijkers, dtype: float64


(61405, 80)

In [34]:
# Eerst omzetten naar strings, daarna komma's verwijderen, en uiteindelijk naar floats
for col in ['KijkersLag1', 'KijkersLag2', 'KijkersLag3', 'Programma']:
    cleanData[col] = cleanData[col].astype(str).str.replace(',', '').astype(float)

# Print het resultaat
print(cleanData[['Kijkers', 'KijkersLag1', 'KijkersLag2', 'KijkersLag3', 'Programma']].head(10))


   Kijkers    KijkersLag1    KijkersLag2    KijkersLag3    Programma
0   721850  892009.049065  892009.049065  892009.049065  1677.000000
1   709606  628581.039823  628581.039823  628581.039823  1425.000000
2   548239  478421.179487  478421.179487  478421.179487  4805.292166
3   523610  791870.406663  791870.406663  791870.406663  1878.000000
4   496216  491224.740741  491224.740741  491224.740741   779.215129
5   447427  351454.000000  351454.000000  351454.000000  2410.568875
6   424041  585860.648837  585860.648837  585860.648837  2553.000000
7   369066  408691.320893  408691.320893  408691.320893  1676.000000
8   368549  336230.500000  336230.500000  336230.500000  2884.658487
9   360544  332870.588235  332870.588235  332870.588235  3290.702201


In [35]:
#opslaan bij models
targetModelDir = os.getcwd()
targetModelPath = os.path.join(targetModelDir, ".",
                          "models", "oneHotTarget.pkl")
os.makedirs(os.path.dirname(targetModelPath), exist_ok=True)
with open(targetModelPath, 'wb') as m:
  pickle.dump(targetEncoding, m)

## Model testing

In [36]:
def scoreModel(model,X,y, cv=3):
    print(f"Model: {model.__class__.__name__}")
    start = time.time()
        
    # MAE
    mae_scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error')
    mae = -np.mean(mae_scores)

    # MAPE
    mape_scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_percentage_error')
    mape = -np.mean(mape_scores) * 100

    duration = time.time() - start
    min, sec = divmod(duration, 60)

    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"Tijd: {int(min)}m {sec:.1f}s")

    return {'Model': model.__class__.__name__, 'MAE': mae, 'MAPE': mape, 'Tijd': duration}
def startModel(models, X, y, cv=3):
    results = []
    for model in models:
        result = scoreModel(model, X, y, cv=cv)
        results.append(result)
    return results

In [37]:
linear = LinearRegression()
randomForest = RandomForestRegressor()
xgb = XGBRegressor()
extraTrees = ExtraTreesRegressor()
histGB = HistGradientBoostingRegressor()
lgbm = LGBMRegressor()

models = [
    RandomForestRegressor(verbose=True), LGBMRegressor(), XGBRegressor(),
    ExtraTreesRegressor(), HistGradientBoostingRegressor(), LinearRegression(),
]

In [38]:
scaler = StandardScaler()
X = dataOneHotTarget.drop(columns=['Kijkers', 'Unnamed: 0'])
X_scaled = scaler.fit_transform(X)
y = dataOneHotTarget['Kijkers']

#trainset en testset maken
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

In [39]:
startModel(models, X, y)

Model: RandomForestRegressor


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   36.5s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:  1.3min finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   37.2s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:  1.3min finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   36.0s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:  1.2min finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   37.1s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:  1.3min finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: 

MAE:  56860.1458
MAPE: 17.32%
Tijd: 7m 31.1s
Model: LGBMRegressor
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2821
[LightGBM] [Info] Number of data points in the train set: 40936, number of used features: 60
[LightGBM] [Info] Start training from score 438430.039574
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2812
[LightGBM] [Info] Number of data points in the train set: 40937, number of used features: 58
[LightGBM] [Info] Start training from score 435732.406942
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001033 seconds.
You can 

[{'Model': 'RandomForestRegressor',
  'MAE': np.float64(56860.145772363314),
  'MAPE': np.float64(17.321210353817516),
  'Tijd': 451.0623893737793},
 {'Model': 'LGBMRegressor',
  'MAE': np.float64(57571.70157542345),
  'MAPE': np.float64(16.552843314512682),
  'Tijd': 3.2601797580718994},
 {'Model': 'XGBRegressor',
  'MAE': np.float64(59813.8203125),
  'MAPE': np.float64(16.87610298395157),
  'Tijd': 2.1758923530578613},
 {'Model': 'ExtraTreesRegressor',
  'MAE': np.float64(56530.43031470621),
  'MAPE': np.float64(16.520010950388897),
  'Tijd': 258.77494859695435},
 {'Model': 'HistGradientBoostingRegressor',
  'MAE': np.float64(57274.92256014476),
  'MAPE': np.float64(16.553079129371774),
  'Tijd': 5.451085329055786},
 {'Model': 'LinearRegression',
  'MAE': np.float64(69396.45159339647),
  'MAPE': np.float64(20.742392699456747),
  'Tijd': 1.205596923828125}]

## Model Performance Comparison

| Model                          | MAE          | MAPE    | Tijd     |
|-------------------------------|--------------|---------|----------|
| **RandomForestRegressor**     | 56,860.15    | 17.32%  | 7m 31.1s |
| **LGBMRegressor**             | 57,571.70    | 16.55%  | 0m 3.3s  |
| **XGBRegressor**              | 59,813.82    | 16.88%  | 0m 2.2s  |
| **ExtraTreesRegressor**       | 56,530.43    | 16.52%  | 4m 18.8s |
| **HistGradientBoostingRegressor** | 57,274.92 | 16.55%  | 0m 5.5s  |
| **LinearRegression**          | 69,396.45    | 20.74%  | 0m 1.2s  |


## Finetuning best models

### RandomizedSearch

extraTrees

In [57]:
et_param = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 50),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.3, 0.7) # Percentage van features om te overwegen bij splitsing
}

In [58]:
rs_et = RandomizedSearchCV(
    estimator=extraTrees,
    param_distributions=et_param,
    n_iter=250,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

start = time.time()
rs_et.fit(X_train, y_train)
duration = time.time() - start

rs_bestET = rs_et.best_estimator_

print("Random Search ExtraTreesRegressor:")
print(f"Tijd: {duration:.2f}s")
print(f"Beste hyperparameters: {rs_et.best_params_}")
print(f"Beste MAE: {rs_et.best_score_:.4f}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Random Search ExtraTreesRegressor:
Tijd: 25872.96s
Beste hyperparameters: {'max_depth': 39, 'max_features': np.float64(0.6233096559497611), 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 875}
Beste MAE: -50149.7415


histGradient

In [59]:
hgb_param = {
    'max_iter': randint(100, 1000),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 15),
    'min_samples_leaf': randint(1, 20),
    'max_leaf_nodes': randint(15, 255),
    'l2_regularization': uniform(0.0, 10.0)
}

In [60]:
rs_hgb = RandomizedSearchCV(
    estimator=histGB,
    param_distributions=hgb_param,
    n_iter=250,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

start = time.time()
rs_hgb.fit(X_train, y_train)
duration = time.time() - start

rs_bestHGB = rs_hgb.best_estimator_

print("Random Search HistGradientBoostingRegressor:")
print(f"Tijd: {duration:.2f}s")
print(f"Beste hyperparameters: {rs_hgb.best_params_}")
print(f"Beste MAE: {rs_hgb.best_score_:.4f}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Random Search HistGradientBoostingRegressor:
Tijd: 2192.67s
Beste hyperparameters: {'l2_regularization': np.float64(5.018049055811807), 'learning_rate': np.float64(0.0193609978629972), 'max_depth': 10, 'max_iter': 871, 'max_leaf_nodes': 128, 'min_samples_leaf': 8}
Beste MAE: -50229.3267


LightGBM

In [45]:
rs_param = {
    'n_estimators': randint(100, 1000),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 15),
    'min_child_weight': randint(1, 10),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.5),
    'reg_lambda': uniform(0.1, 10),
    'reg_alpha': uniform(0, 10)
}

In [ ]:
rs = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=rs_param,
    n_iter=250,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

# Training
start = time.time()
rs.fit(X_train, y_train)
duration = time.time() - start

rs_bestModel = rs.best_estimator_

# Resultaten
print("Random Search LGBMRegressor:")
print(f"Tijd: {duration:.2f}s")
print(f"Beste hyperparameters: {rs.best_params_}")
print(f"Beste MAE: {rs.best_score_:.4f}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001972 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2821
[LightGBM] [Info] Number of data points in the train set: 49124, number of used features: 60
[LightGBM] [Info] Start training from score 446168.762519
Random Search LGBMRegressor:
Tijd: 834.44s
Beste hyperparameters: {'colsample_bytree': np.float64(0.6546041198491748), 'gamma': np.float64(0.12516905444567), 'learning_rate': np.float64(0.06958570693213013), 'max_depth': 12, 'min_child_weight': 8, 'n_estimators': 938, 'reg_alpha': np.float64(3.599078818039688), 'reg_lambda': np.float64(0.955875049311916), 'subsample': np.float64(0.7776493373037157)}
Beste MAE: -50140.7276


### Optuna

extraTrees

In [63]:
def optuna_param(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', [ 'sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
    }

In [64]:
def run_optuna(trial):
    params = optuna_param(trial)
    model = ExtraTreesRegressor(**params, random_state=42, n_jobs=-1)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    return -np.mean(scores)

# Optuna study uitvoeren
start = time.time()

opt = optuna.create_study(direction='minimize')
opt.optimize(run_optuna, n_trials=250, show_progress_bar=True)

# Beste model trainen met optimale parameters
optBest = opt.best_params
optBest = ExtraTreesRegressor(**optBest, random_state=42, n_jobs=-1)
optBest.fit(X_train, y_train)

# Resultaten tonen
duration = time.time() - start

print("Optuna ExtraTreesRegressor:")
print(f"Tijd: {duration:.2f}s")
print(f"Beste hyperparameters: {opt.best_params}")
print(f"Beste MAE: {opt.best_value:.4f}")

  0%|          | 0/250 [00:00<?, ?it/s]

Best trial: 114. Best value: 52786.4: 100%|██████████| 250/250 [1:49:20<00:00, 26.24s/it]


Optuna ExtraTreesRegressor:
Tijd: 6569.88s
Beste hyperparameters: {'n_estimators': 507, 'max_depth': 39, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False}
Beste MAE: 52786.4352


LightGBM

In [73]:
def optuna_param(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10)
    }

In [74]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def run_optuna(trial):
    params = optuna_param(trial)
    model = LGBMRegressor(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    return -np.mean(scores)

# Optuna study uitvoeren
start = time.time()

opt = optuna.create_study(direction='minimize')
opt.optimize(run_optuna, n_trials=500, show_progress_bar=True)

optBestLGBM = opt.best_params
optBestLGBM = LGBMRegressor(**optBestLGBM)
optBestLGBM.fit(X_train, y_train)

duration = time.time() - start

# Resultaten tonen
print("Optuna LGBMRegressor:")
print(f"Tijd: {duration:.2f}s")
print(f"Beste hyperparameters: {opt.best_params}")
print(f"Beste MAE: {opt.best_value:.4f}")

Best trial: 358. Best value: 49886.7: 100%|██████████| 500/500 [43:16<00:00,  5.19s/it]


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001474 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2821
[LightGBM] [Info] Number of data points in the train set: 49124, number of used features: 60
[LightGBM] [Info] Start training from score 446168.762519
Optuna LGBMRegressor:
Tijd: 2598.19s
Beste hyperparameters: {'n_estimators': 981, 'learning_rate': 0.08550615893139353, 'max_depth': 12, 'min_child_weight': 1, 'subsample': 0.8975198522354628, 'colsample_bytree': 0.7387911306240579, 'reg_lambda': 7.729452261127812, 'reg_alpha': 3.869371219839859}
Beste MAE: 49886.7322


HistgradientBoostingRegressor

In [ ]:
def optuna_param(trial):
    return {
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 255),
        'l2_regularization': trial.suggest_float('l2_regularization', 0.0, 10.0),
        'early_stopping': True 
    }

In [56]:
def run_optuna(trial):
    params = optuna_param(trial)
    model = HistGradientBoostingRegressor(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    return -np.mean(scores)

start = time.time()

opt = optuna.create_study(direction='minimize')
opt.optimize(run_optuna, n_trials=250, show_progress_bar=True)

optBestParams = opt.best_params
optBest = HistGradientBoostingRegressor(**optBestParams)
optBest.fit(X_train, y_train)

duration = time.time() - start

print("Optuna HistGradientBoostingRegressor:")
print(f"Duration: {duration:.2f}s")
print(f"Best hyperparameters: {opt.best_params}")
print(f"Best MAE: {opt.best_value:.4f}")

Best trial: 136. Best value: 50027.6: 100%|██████████| 250/250 [3:39:36<00:00, 52.71s/it]   


Optuna HistGradientBoostingRegressor:
Duration: 13195.27s
Best hyperparameters: {'max_iter': 894, 'learning_rate': 0.028126700515035654, 'max_depth': 15, 'min_samples_leaf': 1, 'max_leaf_nodes': 153, 'l2_regularization': 4.3891882015212085}
Best MAE: 50027.6394


## Best model opslaan

Beste model is LGBMRegressor met optuna:
- MAE: 49886.7322
- Beste hyperparameters: {'n_estimators': 981, 'learning_rate': 0.08550615893139353, 'max_depth': 12, 'min_child_weight': 1, 'subsample': 0.8975198522354628, 'colsample_bytree': 0.7387911306240579, 'reg_lambda': 7.729452261127812, 'reg_alpha': 3.869371219839859}


In [75]:
with open('./models/optunaBestModel.pkl', 'wb') as f:
    pickle.dump(optBestLGBM, f)